# tolvaptan_study: 症状データの結合

- 入力: `tolvap_sym_manual(01).csv`
- 結合キー: `Patient_ID` のみ
- 追加列: `HFonset` (TEXT, YYYY-MM-DD), `weight_gain` / `edema` / `dyspnea` (INTEGER, 1=あり 0=なし), `コメント` (TEXT)
- 既存列がある場合はその列をスキップ（上書きしない）
- 実行前にDBをバックアップ

In [4]:
import sqlite3
from datetime import datetime
from pathlib import Path

import pandas as pd

CSV_PATH = Path('/Users/muna/Hana_research/data/processed/tolvaptan/tolvap_sym_manual(01).csv')
DB_PATH = Path('/Users/muna/Hana_research/data/db/Hana_Research.db')
TABLE = 'tolvaptan_study'
KEY = 'Patient_ID'
NEW_COLS = {
    'HFonset': 'TEXT',
    'weight_gain': 'INTEGER',
    'edema': 'INTEGER',
    'dyspnea': 'INTEGER',
    'コメント': 'TEXT',
}
FLAG_COLS = ['weight_gain', 'edema', 'dyspnea']

## 1. DBバックアップ

In [ ]:
assert DB_PATH.exists(), f'DB not found: {DB_PATH}'
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
BACKUP_PATH = DB_PATH.with_name(f'{DB_PATH.stem}_backup_{stamp}{DB_PATH.suffix}')

src = sqlite3.connect(DB_PATH)
dst = sqlite3.connect(BACKUP_PATH)
try:
    src.backup(dst)
finally:
    dst.close()
    src.close()
print('backup:', BACKUP_PATH)

## 2. CSV読み込みと検証

In [5]:
def read_csv_auto(path):
    for enc in ('utf-8-sig', 'cp932'):
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc)
            print('encoding:', enc)
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError('Encoding is neither UTF-8 nor Shift_JIS (cp932).')


raw = read_csv_auto(CSV_PATH)
# 行末カンマ由来の空列を除外
raw = raw.loc[:, ~raw.columns.str.startswith('Unnamed')]
raw.columns = raw.columns.str.strip()

need = [KEY, *NEW_COLS]
missing_cols = [c for c in need if c not in raw.columns]
assert not missing_cols, f'CSVに列がありません: {missing_cols}'

df = raw[need].copy()
# 欠損チェックは コメント以外の列に限定（コメント空欄は正常値）
na_counts = df[[KEY, 'HFonset', *FLAG_COLS]].isna().sum()
assert na_counts.sum() == 0, f'欠損あり:\n{na_counts[na_counts > 0]}'

df[KEY] = df[KEY].str.strip().astype(int)
assert df[KEY].is_unique, 'Patient_ID が重複しています'

df['HFonset'] = (
    pd.to_datetime(df['HFonset'].str.strip(), format='%Y-%m-%d')
    .dt.strftime('%Y-%m-%d')
)
for c in FLAG_COLS:
    df[c] = df[c].str.strip().astype(int)
    assert df[c].isin([0, 1]).all(), f'{c} に 0/1 以外の値があります'

# コメント: 前後空白を除去し、空欄は None (NULL) にする
df['コメント'] = df['コメント'].str.strip()
df['コメント'] = df['コメント'].where(df['コメント'].notna() & (df['コメント'] != ''), None)
print('コメント空欄 (NULL):', df['コメント'].isna().sum(), '件')

print('CSV rows:', len(df))
df.head()

encoding: utf-8-sig
コメント空欄 (NULL): 31 件
CSV rows: 159


,Patient_ID,HFonset,weight_gain,edema,dyspnea,コメント
0,150004,2017-12-01,1,1,1,ラシ＋
1,150009,2016-11-07,1,1,0,アル＋
2,150041,2016-04-19,1,1,0,ダイア＋，BNP
3,150067,2015-08-07,0,1,1,胸水＋
4,150068,2018-02-14,0,1,0,ラシ＋


## 3. DBとの突合チェック

In [6]:
con = sqlite3.connect(DB_PATH)

db_ids = pd.read_sql_query(f'SELECT [{KEY}] FROM [{TABLE}]', con)[KEY]
not_in_db = sorted(set(df[KEY]) - set(db_ids))
if not_in_db:
    con.close()
    raise ValueError(f'CSVにあってDBにない Patient_ID: {not_in_db}')

print('DB rows        :', len(db_ids))
print('CSV rows       :', len(df))
print('DB only (NULL) :', len(set(db_ids) - set(df[KEY])))

DB rows        : 214
CSV rows       : 159
DB only (NULL) : 55


## 4. 列追加と値の書き込み

In [7]:
existing = {r[1] for r in con.execute(f'PRAGMA table_info([{TABLE}])')}
to_add = {c: t for c, t in NEW_COLS.items() if c not in existing}
skipped = [c for c in NEW_COLS if c in existing]
print('add    :', list(to_add))
print('skip   :', skipped)

if to_add:
    for c, t in to_add.items():
        con.execute(f'ALTER TABLE [{TABLE}] ADD COLUMN [{c}] {t}')

    cols = list(to_add)
    set_clause = ', '.join(f'[{c}] = ?' for c in cols)
    sql = f'UPDATE [{TABLE}] SET {set_clause} WHERE [{KEY}] = ?'
    rows = df[cols + [KEY]].astype(object).values.tolist()
    with con:
        cur = con.executemany(sql, rows)
    print('updated rows:', cur.rowcount)

add    : ['HFonset', 'weight_gain', 'edema', 'dyspnea', 'コメント']
skip   : []
updated rows: 159


## 5. 検証

In [8]:
if to_add:
    cols = list(to_add)
    col_sql = ', '.join(f'[{c}]' for c in cols)
    out = pd.read_sql_query(f'SELECT [{KEY}], {col_sql} FROM [{TABLE}]', con)

    merged = df[[KEY] + cols].merge(out, on=KEY, suffixes=('_csv', '_db'), validate='one_to_one')
    for c in cols:
        a, b = merged[f'{c}_csv'], merged[f'{c}_db']
        # NULL同士は一致とみなす
        same = (a == b) | (a.isna() & b.isna())
        n_diff = int((~same).sum())
        print(f'{c}: CSVとDBの不一致 {n_diff} 件')
        assert n_diff == 0

    print()
    print('非NULL件数 (CSV行数:', len(df), ')')
    print(out[cols].notna().sum())
else:
    print('追加列なし（全列スキップ）')

print()
print([r[1] for r in con.execute(f'PRAGMA table_info([{TABLE}])')])
con.close()

HFonset: CSVとDBの不一致 0 件
weight_gain: CSVとDBの不一致 0 件
edema: CSVとDBの不一致 0 件
dyspnea: CSVとDBの不一致 0 件
コメント: CSVとDBの不一致 0 件

非NULL件数 (CSV行数: 159 )
HFonset        159
weight_gain    159
edema          159
dyspnea        159
コメント           128
dtype: int64

['Patient_ID', 'Study_ID', 'ind_date', 'date_precision', 'raw_match', 'ef_visit_date', 'ef_value', 'ef_matched_text', 'ef_final_hf_class', 'HF', 'Cancer', 'CKD', 'Cirrhosis', 'HypoNa', 'pre_weight', 'pre_weight_date', 'HFonset', 'weight_gain', 'edema', 'dyspnea', 'コメント']
